# M01-02 Sesión Spark y primer Data Frame
# Paso 1 Arranque

Localizo el repo y las rutas para ver si todo esta correctamente configurado y en tal caso arranca el siguiente paso.



In [6]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)

ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated


# Paso 2 - Comprueba Java y PySpark

Es necesario comprobar las versiones de Java, es requerido que tenga JRE 17 para que spark arranque.



In [7]:
import pyspark, shutil, subprocess

print("pyspark", pyspark.__version__)
print(subprocess.check_output(["java", "-version"], text=True, stderr=subprocess.STDOUT).splitlines()[0])
print("java:", shutil.which("java"))


pyspark 3.5.5
openjdk version "17.0.20.1" 2026-08-18
java: /usr/lib/jvm/java-17-openjdk-amd64/bin/java


# Paso 3 Crea o reutiliza la sesión



En primer lugar se va a pedir la sesion de Spark para evitar crear un segundo contexto


In [8]:
spark = get_spark("novashop-m01")
spark

# Paso 4 - Cinco pedidos en memoria

Creamos un dataframe, es una tabla sin ficheros.

In [9]:
from pyspark.sql import Row

pedidos = [
    Row(order_id="O90001", customer_id="C0001", status="paid", amount=49.90),
    Row(order_id="O90002", customer_id="C0002", status="paid", amount=12.50),
    Row(order_id="O90003", customer_id="C0003", status="cancelled", amount=80.00),
    Row(order_id="O90004", customer_id="C0001", status="paid", amount=23.10),
    Row(order_id="O90005", customer_id="C0004", status="pending", amount=5.00),
]
df = spark.createDataFrame(pedidos)
df.printSchema()
df.show()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)

+--------+-----------+---------+------+
|order_id|customer_id|   status|amount|
+--------+-----------+---------+------+
|  O90001|      C0001|     paid|  49.9|
|  O90002|      C0002|     paid|  12.5|
|  O90003|      C0003|cancelled|  80.0|
|  O90004|      C0001|     paid|  23.1|
|  O90005|      C0004|  pending|   5.0|
+--------+-----------+---------+------+



# Paso 5 - Uso de filter y Count

Filter es una extensión que utiliza más memoria alarga el plan, pero no ejectua.

In [11]:
paid = df.filter(df.status == "paid")
print("después del filter, Spark aún no ha contado nada")
print("paid count =", paid.count())
paid.show()

customer = df.filter(df.customer_id == "C0001")
print("Filtro ocn salida == C0001")
print("C0001 count =", customer.count())
customer.show()


después del filter, Spark aún no ha contado nada
paid count = 3
+--------+-----------+------+------+
|order_id|customer_id|status|amount|
+--------+-----------+------+------+
|  O90001|      C0001|  paid|  49.9|
|  O90002|      C0002|  paid|  12.5|
|  O90004|      C0001|  paid|  23.1|
+--------+-----------+------+------+

Filtro ocn salida == C0001
C0001 count = 2
+--------+-----------+------+------+
|order_id|customer_id|status|amount|
+--------+-----------+------+------+
|  O90001|      C0001|  paid|  49.9|
|  O90004|      C0001|  paid|  23.1|
+--------+-----------+------+------+



# Mejora

Uso de withColumn

In [12]:
from pyspark.sql.functions import lit

df.withColumn("channel", lit("web")).select("order_id", "channel").show()


+--------+-------+
|order_id|channel|
+--------+-------+
|  O90001|    web|
|  O90002|    web|
|  O90003|    web|
|  O90004|    web|
|  O90005|    web|
+--------+-------+



## M02 Arrancamos ahora la fase de ingesta y preparación 

